In [1]:
import os
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
folder = "ifpi"

In [3]:
def seznam_souboru(slozka):
    seznam = []
    for file in os.listdir(slozka):
        seznam.append(file)
    return(seznam)

In [4]:
def scrapuj_hitparadu(slozka, soubor):
    with open(os.path.join(slozka, soubor), "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")
        hitparada = soup.find("h1").text.strip()
        obdobi = soup.find(class_="header__subtitle").text.strip()
        tracky = soup.find_all(class_="chart-full-row")
        zebricek = []
        for t in tracky:
            track = {}
            track['hitparada'] = hitparada
            track['obdobi'] = obdobi
            track['pozice'] = int(t.find(class_='chart-full-row__position').text.strip().split(".")[0])
            track['track/album'] = t.find(class_="chart-full-row__title").text.strip()
            interpretstvo = t.find(class_="chart-full-row__interpreter").text.strip().split("\n")
            track['interpret'] = interpretstvo[0]
            track['vydavatel'] = interpretstvo[-1][2:]
            
            odkazy = t.find(class_="play-dropdown__menu")
            if odkazy != None:
                odkazy = odkazy.find_all("a")
                for o in odkazy:
                    if o.text.lower() == "youtube":
                        track['youtube'] = o['href']
            
            track['soubor'] = soubor
            zebricek.append(track)
    return(zebricek)

In [7]:
import time

In [9]:
kolik = 30

In [11]:
import os

if not os.path.exists("data_raw"):
    os.mkdir("data_raw")
else:
    pass

In [13]:
import json

In [15]:
from IPython.display import clear_output

In [17]:
def scrapuj_hitparady(slozka, kolik = None):
    count = 0 
    start_time = time.time()
    ke_zpracovani = [x for x in seznam_souboru(slozka)]
    for s in ke_zpracovani[:kolik]:
        time_taken = time.time() - start_time
        count = count + 1
        time_remaining = (time_taken / count) * (len(ke_zpracovani[:kolik]) - count)
        if time_remaining >= 1:
            time_remaining = int(time_remaining)
        else:
            time_remaining = "--"
        print(f"done: {count} / total: {len(ke_zpracovani[:kolik])} / ETR: {time_remaining} s / {s}")
        h = scrapuj_hitparadu(slozka, s)
        with open(os.path.join("data_raw",s.replace(".html",".json")), "w") as file:
            json.dump(h, file)
        clear_output(wait=True)

In [21]:
scrapuj_hitparady(folder)

done: 1140 / total: 1140 / ETR: -- s / 7-2966.html
